# Deteksi Transaksi Keuangan Mencurigakan
Perbandingan Decision Tree dan Random Forest. Unggah CSV ke panel Files Colab sebelum menjalankan semua sel.

In [ ]:
# Deteksi Transaksi Keuangan Mencurigakan
# Decision Tree vs Random Forest
# Unggah CSV dengan nama C02_Deteksi_Transaksi_Keuangan_Mencurigakan.csv ke panel Files Colab.

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.base import clone

# 1. Memuat dan mengeksplorasi data
df = pd.read_csv('/content/C02_Deteksi_Transaksi_Keuangan_Mencurigakan.csv')
print("Ukuran data:", df.shape)
display(df.head())
df.info()
display(df['Status_Transaksi'].value_counts(dropna=False))
display(df.isna().sum().to_frame('Jumlah nilai kosong'))
print("Duplikasi:", df.duplicated().sum())
display(df.describe(include='all').T)

# 2. Fitur dan target (1=Mencurigakan, 0=Normal)
TARGET = 'Status_Transaksi'
X = df.drop(columns=[TARGET])
y = (df[TARGET].astype(str).str.strip().str.lower() == 'mencurigakan').astype(int)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# 3. Preprocessing di dalam pipeline: median, modus, One-Hot Encoding
# Ini mencegah data leakage karena parameter preprocessing dipelajari dari data latih.
preprocess = ColumnTransformer([
    ('numerik', SimpleImputer(strategy='median'), num_cols),
    ('kategori', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

# 4. Split stratified 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

# 5. Delapan skenario parameter
scenarios = [
 ('Decision Tree','DT-1','Gini; depth=None; leaf=1; weight=None',
  DecisionTreeClassifier(criterion='gini',max_depth=None,min_samples_leaf=1,random_state=42)),
 ('Decision Tree','DT-2','Gini; depth=5; leaf=2; weight=None',
  DecisionTreeClassifier(criterion='gini',max_depth=5,min_samples_leaf=2,random_state=42)),
 ('Decision Tree','DT-3','Entropy; depth=10; leaf=5; weight=balanced',
  DecisionTreeClassifier(criterion='entropy',max_depth=10,min_samples_leaf=5,class_weight='balanced',random_state=42)),
 ('Decision Tree','DT-4','Entropy; depth=15; leaf=10; weight=None',
  DecisionTreeClassifier(criterion='entropy',max_depth=15,min_samples_leaf=10,random_state=42)),
 ('Random Forest','RF-1','trees=100; depth=None; leaf=1; weight=None',
  RandomForestClassifier(n_estimators=100,max_depth=None,min_samples_leaf=1,random_state=42,n_jobs=-1)),
 ('Random Forest','RF-2','trees=200; depth=None; leaf=1; weight=balanced',
  RandomForestClassifier(n_estimators=200,max_depth=None,min_samples_leaf=1,class_weight='balanced',random_state=42,n_jobs=-1)),
 ('Random Forest','RF-3','trees=200; depth=10; leaf=2; weight=None',
  RandomForestClassifier(n_estimators=200,max_depth=10,min_samples_leaf=2,random_state=42,n_jobs=-1)),
 ('Random Forest','RF-4','trees=300; depth=15; leaf=5; weight=balanced_subsample',
  RandomForestClassifier(n_estimators=300,max_depth=15,min_samples_leaf=5,class_weight='balanced_subsample',random_state=42,n_jobs=-1))
]

# 6. Training dan evaluasi pada test set yang sama
records, pipelines = [], {}
for algorithm, code, setting, model in scenarios:
    pipe = Pipeline([('preprocess', clone(preprocess)), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0,1]).ravel()
    records.append({'Algoritma':algorithm,'Kode':code,'Setting':setting,
        'Accuracy':accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred,zero_division=0),
        'Recall':recall_score(y_test,pred,zero_division=0),
        'F1':f1_score(y_test,pred,zero_division=0),
        'ROC_AUC':roc_auc_score(y_test,prob),'TN':tn,'FP':fp,'FN':fn,'TP':tp})
    pipelines[code] = pipe

hasil = pd.DataFrame(records).sort_values(['F1','Recall','ROC_AUC'],ascending=False)
display(hasil.round(4))

# 7. Model terbaik: F1 tertinggi, kemudian Recall dan ROC-AUC
best_code = hasil.iloc[0]['Kode']
best_model = pipelines[best_code]
pred = best_model.predict(X_test)
print("Model terbaik:", best_code)
print(classification_report(y_test,pred,target_names=['Normal','Mencurigakan']))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['Normal','Mencurigakan'])
plt.show()

hasil.sort_values('Kode').plot(x='Kode',y=['Accuracy','Precision','Recall','F1'],kind='bar',figsize=(12,5))
plt.ylim(0,1); plt.title('Perbandingan metrik eksperimen'); plt.tight_layout(); plt.show()
hasil.to_csv('/content/hasil_eksperimen.csv',index=False)
print("Hasil tersimpan sebagai hasil_eksperimen.csv")
